⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "ipython",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "nbconvert_exporter": "python",
      "pygments_lexer": "ipython3",
      "version": "3.10.12"
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "# ODI to Databricks PySpark Migration\n",
        "\n",
        "**Source File:** ODI Script - `SCEN_TASK_NO` in `{1} - {210}`\n",
        "**Conversion Date:** 2023-10-27"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from delta.tables import DeltaTable\n",
        "from pyspark.sql import functions as F\n",
        "from pyspark.sql.types import (\n",
        "    StructType, StructField,\n",
        "    StringType, LongType, IntegerType, DoubleType,\n",
        "    DecimalType, TimestampType, DateType, BinaryType, FloatType\n",
        ")\n",
        "from pyspark.sql.window import Window\n",
        "from pyspark.sql import SparkSession\n",
        "\n",
        "spark = SparkSession.builder.getOrCreate()"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\")\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\",      \"\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\",       \"\")\n",
        "dbutils.widgets.text(\"v_ETL_JOB_TYPE\",    \"\")\n",
        "\n",
        "datasource_num_id = int(dbutils.widgets.get(\"DATASOURCE_NUM_ID\"))\n",
        "etl_proc_wid      = int(dbutils.widgets.get(\"ETL_PROC_WID\"))\n",
        "odi_sess_no       = dbutils.widgets.get(\"ODI_SESS_NO\")\n",
        "v_etl_job_type    = dbutils.widgets.get(\"v_ETL_JOB_TYPE\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## ETL Parameters"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# SCEN_TASK_NO {2}, {3}, {4}, {5}, {6}\n",
        "etl_parameters_df = (\n",
        "    spark.table(\"workspace.prxbi_dw.wc_etl_parameters\")\n",
        "    .filter(F.col(\"ETL_JOB_TYPE\") == F.lit(v_etl_job_type))\n",
        ")\n",
        "\n",
        "etl_last_extract_time = (\n",
        "    etl_parameters_df\n",
        "    .agg(F.max(\"etl_last_extract_time\").alias(\"val\"))\n",
        "    .collect()[0][\"val\"]\n",
        ")\n",
        "\n",
        "etl_current_extract_time = (\n",
        "    etl_parameters_df\n",
        "    .agg(F.max(\"etl_current_extract_time\").alias(\"val\"))\n",
        "    .collect()[0][\"val\"]\n",
        ")\n",
        "\n",
        "etl_row_wid = (\n",
        "    etl_parameters_df\n",
        "    .agg(F.max(\"ROW_WID\").alias(\"val\"))\n",
        "    .collect()[0][\"val\"]\n",
        ")\n",
        "\n",
        "print(f\"ETL Last Extract Time: {etl_last_extract_time}\")\n",
        "print(f\"ETL Current Extract Time: {etl_current_extract_time}\")\n",
        "print(f\"ETL Row WID: {etl_row_wid}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Staging Table: `c_mercury_badge_ts_stg`"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# SCEN_TASK_NO {30}\n",
        "spark.sql(\"DROP TABLE IF EXISTS workspace.prxbi_dw.c_mercury_badge_ts_stg PURGE\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# SCEN_TASK_NO {40}, {50}\n",
        "source_mercury_badge_ts_df = spark.table(\"workspace.prxbi_ts.wc_mercury_badge_ts\")\n",
        "\n",
        "# Apply filter and deduplication for staging data\n",
        "# ODI pattern: INNER JOIN (SELECT key, MAX(col1), MAX(col2) ... GROUP BY key)\n",
        "# Converted to Window function as per rule P.12\n",
        "window_spec_staging = Window.partitionBy(F.col(\"ID\")).orderBy(\n",
        "    F.col(\"INT_INSERT_DATE\").desc(),\n",
        "    F.col(\"VERSIONNUMBER\").desc()\n",
        ")\n",
        "\n",
        "c_mercury_badge_ts_stg_df = (\n",
        "    source_mercury_badge_ts_df\n",
        "    .filter(\n",
        "        (F.col(\"INT_INSERT_DATE\") > F.lit(etl_last_extract_time)) &\n",
        "        (F.col(\"INT_INSERT_DATE\") <= F.lit(etl_current_extract_time))\n",
        "    )\n",
        "    .withColumn(\"rn\", F.row_number().over(window_spec_staging))\n",
        "    .filter(F.col(\"rn\") == 1)\n",
        "    .drop(\"rn\")\n",
        "    .select(\n",
        "        F.col(\"ID\").cast(StringType()).alias(\"ID\"),\n",
        "        F.col(\"BADGELOCATION\").cast(StringType()).alias(\"BADGELOCATION\"),\n",
        "        F.col(\"BADGETOKEN\").cast(StringType()).alias(\"BADGETOKEN\"),\n",
        "        F.col(\"BADGEVERSION\").cast(LongType()).alias(\"BADGEVERSION\"),\
        "        F.col(\"CONTACTEMAIL\").cast(StringType()).alias(\"CONTACTEMAIL\"),\n",
        "        F.col(\"CONTACTFIRSTNAME\").cast(StringType()).alias(\"CONTACTFIRSTNAME\"),\n",
        "        F.col(\"CONTACTJOBTITLE\").cast(StringType()).alias(\"CONTACTJOBTITLE\"),\n",
        "        F.col(\"CONTACTLASTNAME\").cast(StringType()).alias(\"CONTACTLASTNAME\"),\n",
        "        F.col(\"CONTACTPERSONRXMASTERID\").cast(StringType()).alias(\"CONTACTPERSONRXMASTERID\"),\n",
        "        F.col(\"CREATEDBYREGISTRATIONTYPE\").cast(StringType()).alias(\"CREATEDBYREGISTRATIONTYPE\"),\
        "        F.col(\"CREATEDBYTYPE\").cast(StringType()).alias(\"CREATEDBYTYPE\"),\n",
        "        F.col(\"CULTURE\").cast(StringType()).alias(\"CULTURE\"),\n",
        "        F.col(\"CUSTOMERTYPE\").cast(StringType()).alias(\"CUSTOMERTYPE\"),\n",
        "        F.col(\"EVENTEDITIONGBSCODE\").cast(StringType()).alias(\"EVENTEDITIONGBSCODE\"),\
        "        F.col(\"ISBADGEUPDATE\").cast(StringType()).alias(\"ISBADGEUPDATE\"),\n",
        "        F.col(\"MARKETINGPREFERENCESPROMPTREQUIRED\").cast(StringType()).alias(\"MARKETINGPREFERENCESPROMPTREQU\"),\n",
        "        F.col(\"ORGANISATIONCITY\").cast(StringType()).alias(\"ORGANISATIONCITY\"),\n",
        "        F.col(\"ORGANISATIONCOUNTRYCODE\").cast(StringType()).alias(\"ORGANISATIONCOUNTRYCODE\"),\
        "        F.col(\"ORGANISATIONDISPLAYNAME\").cast(StringType()).alias(\"ORGANISATIONDISPLAYNAME\"),\
        "        F.col(\"ORGANISATIONRXMASTERID\").cast(StringType()).alias(\"ORGANISATIONRXMASTERID\"),\n",
        "        F.col(\"ORGANISATIONSTATE\").cast(StringType()).alias(\"ORGANISATIONSTATE\"),\n",
        "        F.col(\"PARTICIPATINGORGANISATIONID\").cast(StringType()).alias(\"PARTICIPATINGORGANISATIONID\"),\
        "        F.col(\"PRODUCTCODE\").cast(StringType()).alias(\"PRODUCTCODE\"),\n",
        "        F.col(\"QRCODECONTENT\").cast(StringType()).alias(\"QRCODECONTENT\"),\n",
        "        F.col(\"REGISTRATIONID\").cast(StringType()).alias(\"REGISTRATIONID\"),\n",
        "        F.col(\"STATUS\").cast(LongType()).alias(\"STATUS\"),\n",
        "        F.col(\"SUPPORTSTAFFCOMPANYADDRESS\").cast(StringType()).alias(\"SUPPORTSTAFFCOMPANYADDRESS\"),\n",
        "        F.col(\"SUPPORTSTAFFCOMPANYNAME\").cast(StringType()).alias(\"SUPPORTSTAFFCOMPANYNAME\"),\
        "        F.col(\"SUPPORTSTAFFMOBILEPHONE\").cast(StringType()).alias(\"SUPPORTSTAFFMOBILEPHONE\"),\
        "        F.col(\"SUPPORTSTAFFREPORTSTO\").cast(StringType()).alias(\"SUPPORTSTAFFREPORTSTO\"),\n",
        "        F.col(\"SUPPORTSTAFFSTANDS\").cast(StringType()).alias(\"SUPPORTSTAFFSTANDS\"),\n",
        "        F.col(\"SUPPORTSTAFFUSERACCESS\").cast(StringType()).alias(\"SUPPORTSTAFFUSERACCESS\"),\n",
        "        F.col(\"VERSIONNUMBER\").cast(LongType()).alias(\"VERSIONNUMBER\"),\n",
        "        F.col(\"MOBILEPHONE\").cast(StringType()).alias(\"MOBILEPHONE\"),\n",
        "        F.col(\"FIRSTSCANNEDDATE\").cast(TimestampType()).alias(\"FIRSTSCANNEDDATE\"),\n",
        "        F.col(\"LASTPRINTEDDATE\").cast(TimestampType()).alias(\"LASTPRINTEDDATE\"),\n",
        "        F.col(\"ACCESSVALIDITYMODIFIEDDATE\").cast(TimestampType()).alias(\"ACCESSVALIDITYMODIFIEDDATE\"),\
        "        F.col(\"CREATEDDATE\").cast(TimestampType()).alias(\"CREATEDDATE\"),\n",
        "        F.col(\"COMPANYPRODUCTCODE\").cast(StringType()).alias(\"COMPANYPRODUCTCODE\"),\n",
        "        F.col(\"PAYMENTSTATUS\").cast(StringType()).alias(\"PAYMENTSTATUS\"),\n",
        "        F.col(\"PHOTOKEY\").cast(StringType()).alias(\"PHOTOKEY\"),\n",
        "        F.col(\"PHOTOSOURCE\").cast(StringType()).alias(\"PHOTOSOURCE\"),\n",
        "        F.col(\"PHOTOSOURCETYPE\").cast(StringType()).alias(\"PHOTOSOURCETYPE\")\n",
        "    )\n",
        ")\n",
        "\n",
        "c_mercury_badge_ts_stg_df.write.format(\"delta\").mode(\"overwrite\").saveAsTable(\"workspace.prxbi_dw.c_mercury_badge_ts_stg\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "c_mercury_badge_ts_stg_count = spark.table(\"workspace.prxbi_dw.c_mercury_badge_ts_stg\").count()\n",
        "print(f\"Staging table count: {c_mercury_badge_ts_stg_count}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Flow Table: `i_wc_badge_details_d_flow`"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# SCEN_TASK_NO {80}\n",
        "spark.sql(\"DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow PURGE\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# SCEN_TASK_NO {90}, {100}\n",
        "c_staging_df = spark.table(\"workspace.prxbi_dw.c_mercury_badge_ts_stg\")\n",
        "wc_badge_product_d_df = spark.table(\"workspace.prxbi_dw.wc_badge_product_d\")\n",
        "\n",
        "# Deduplicate WC_BADGE_PRODUCT_D as per ODI's rank() over(partition by SKU order by ID desc)\n",
        "window_spec_product = Window.partitionBy(F.col(\"SKU\")).orderBy(F.col(\"ID\").desc())\n",
        "wc_badge_product_d_deduped_df = (\n",
        "    wc_badge_product_d_df\n",
        "    .withColumn(\"col\", F.rank().over(window_spec_product))\n",
        "    .filter(F.col(\"col\") == 1)\n",
        "    .drop(\"col\")\n",
        "    .select(\n",
        "        F.col(\"SKU\").alias(\"SKU_1\"),\n",
        "        F.col(\"NAME\").alias(\"NAME_1\")\n",
        "    )\n",
        ")\n",
        "\n",
        "i_wc_badge_details_d_flow_df = (\n",
        "    c_staging_df.alias(\"JOIN1_A\")\n",
        "    .join(\n",
        "        wc_badge_product_d_deduped_df.alias(\"WC_BADGE_PRODUCT_D_2\"),\n",
        "        F.col(\"JOIN1_A.PRODUCTCODE\") == F.col(\"WC_BADGE_PRODUCT_D_2.SKU_1\"),\n",
        "        \"left_outer\"\n",
        "    )\n",
        "    .select(\n",
        "        F.col(\"JOIN1_A.ID\").alias(\"BADGE_ID\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.BADGELOCATION\").alias(\"BADGE_LOCATION\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.BADGETOKEN\").alias(\"BADGE_TOKEN\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.BADGEVERSION\").alias(\"BADGE_VERSION\").cast(LongType()),\n",
        "        F.col(\"JOIN1_A.CONTACTEMAIL\").alias(\"CONTACT_EMAIL\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.CONTACTFIRSTNAME\").alias(\"CONTACT_FIRST_NAME\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.CONTACTLASTNAME\").alias(\"CONTACT_LAST_NAME\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.CONTACTJOBTITLE\").alias(\"CONTACT_JOB_TITLE\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.CONTACTPERSONRXMASTERID\").alias(\"CONTACT_PERSON_ID\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.CREATEDBYREGISTRATIONTYPE\").alias(\"CREATION_REG_TYPE\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.CREATEDBYTYPE\").alias(\"CREATION_TYPE\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.CULTURE\").alias(\"CULTURE\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.CUSTOMERTYPE\").alias(\"CUSTOMER_TYPE\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.EVENTEDITIONGBSCODE\").alias(\"EVENT_EDITION_CODE\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.ISBADGEUPDATE\").alias(\"BADGE_UPDATE_FLG\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.MARKETINGPREFERENCESPROMPTREQU\").alias(\"MARKETING_PREF_PROMPT\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.ORGANISATIONDISPLAYNAME\").alias(\"ORG_NAME\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.ORGANISATIONCITY\").alias(\"ORG_CITY\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.ORGANISATIONCOUNTRYCODE\").alias(\"ORG_COUNTRY\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.ORGANISATIONRXMASTERID\").alias(\"ORG_ID\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.ORGANISATIONSTATE\").alias(\"ORG_STATE\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.PARTICIPATINGORGANISATIONID\").alias(\"PARTICIPATING_ORG_ID\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.PRODUCTCODE\").alias(\"PRODUCT_CODE\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.QRCODECONTENT\").alias(\"QR_CODE\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.REGISTRATIONID\").alias(\"REGISTRATION_ID\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.STATUS\").alias(\"STATUS\").cast(LongType()),\n",
        "        F.col(\"JOIN1_A.SUPPORTSTAFFCOMPANYNAME\").alias(\"STAFF_COMPANY_NAME\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.SUPPORTSTAFFCOMPANYADDRESS\").alias(\"STAFF_COMPANY_ADDR\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.SUPPORTSTAFFMOBILEPHONE\").alias(\"STAFF_PHONE_NUM\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.SUPPORTSTAFFREPORTSTO\").alias(\"STAFF_REPORTING\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.SUPPORTSTAFFSTANDS\").alias(\"STAFF_STANDS\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.SUPPORTSTAFFUSERACCESS\").alias(\"STAFF_USER_ACCESS\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.VERSIONNUMBER\").alias(\"VERSION_NUM\").cast(LongType()),\n",
        "        F.col(\"JOIN1_A.ID\").alias(\"INTEGRATION_ID\").cast(StringType()),\n",
        "        F.lit(380).cast(StringType()).alias(\"DATASOURCE_NUM_ID\"), # Original VARCHAR2(10 CHAR)\n",
        "        F.col(\"JOIN1_A.MOBILEPHONE\").alias(\"MOBILEPHONE\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.FIRSTSCANNEDDATE\").alias(\"FIRSTSCANNEDDATE\").cast(TimestampType()),\n",
        "        F.col(\"JOIN1_A.LASTPRINTEDDATE\").alias(\"LASTPRINTEDDATE\").cast(TimestampType()),\n",
        "        F.when(F.col(\"JOIN1_A.FIRSTSCANNEDDATE\").isNotNull(), F.lit(\"Y\")).otherwise(F.lit(\"N\")).alias(\"FIRSTSCANNEDDATE_FLG\").cast(StringType()),\n",
        "        F.when(F.col(\"JOIN1_A.LASTPRINTEDDATE\").isNotNull(), F.lit(\"Y\")).otherwise(F.lit(\"N\")).alias(\"LASTPRINTEDDATE_FLG\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.ACCESSVALIDITYMODIFIEDDATE\").alias(\"ACCESSVALIDITYMODIFIEDDATE\").cast(TimestampType()),\n",
        "        F.col(\"JOIN1_A.CREATEDDATE\").alias(\"CREATEDDATE\").cast(TimestampType()),\n",
        "        F.col(\"JOIN1_A.COMPANYPRODUCTCODE\").alias(\"COMPANYPRODUCTCODE\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.PAYMENTSTATUS\").alias(\"PAYMENTSTATUS\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.PHOTOKEY\").alias(\"PHOTOKEY\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.PHOTOSOURCE\").alias(\"PHOTOSOURCE\").cast(StringType()),\n",
        "        F.col(\"JOIN1_A.PHOTOSOURCETYPE\").alias(\"PHOTOSOURCETYPE\").cast(StringType()),\n",
        "        F.col(\"WC_BADGE_PRODUCT_D_2.NAME_1\").alias(\"PACKAGE_NAME\").cast(StringType()),\n",
        "        F.lit(\"I\").alias(\"IND_UPDATE\").cast(StringType())\n",
        "    )\n",
        ")\n",
        "\n",
        "i_wc_badge_details_d_flow_df.write.format(\"delta\").mode(\"overwrite\").saveAsTable(\"workspace.prxbi_dw.i_wc_badge_details_d_flow\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "i_wc_badge_details_d_flow_count = spark.table(\"workspace.prxbi_dw.i_wc_badge_details_d_flow\").count()\n",
        "print(f\"Flow table count: {i_wc_badge_details_d_flow_count}\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# SCEN_TASK_NO {110}, {120}\n",
        "spark.sql(\"SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false\")\n",
        "spark.sql(\"OPTIMIZE workspace.prxbi_dw.i_wc_badge_details_d_flow ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID)\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Error / Audit Tables (Placeholders)"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Placeholder for E$ table creation if needed (not explicitly in source script for E$)\n",
        "# spark.sql(\"CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.e_some_error_table (...)\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Placeholder for deleting session-specific records from E$ table\n",
        "# spark.sql(f\"DELETE FROM workspace.prxbi_dw.e_some_error_table WHERE ODI_SESS_NO = '{odi_sess_no}'\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Placeholder for SNP_CHECK_TAB creation if needed\n",
        "# spark.sql(\"CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.snp_check_tab (CHECK_NAME STRING, CHECK_VALUE LONG, ODI_SESS_NO STRING)\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Placeholder for deleting session-specific records from SNP_CHECK_TAB\n",
        "# spark.sql(f\"DELETE FROM workspace.prxbi_dw.snp_check_tab WHERE ODI_SESS_NO = '{odi_sess_no}'\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## PK Violation Detection (Placeholder)"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Placeholder for inserting duplicates into E$ via DataFrame append\n",
        "# This step is usually handled by the merge's WHEN NOT MATCHED BY SOURCE or explicit duplicate detection before merge.\n",
        "# (Not explicitly present in this ODI script)"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Placeholder for deduplicating flow table - ROW_NUMBER overwrite\n",
        "# The flow table is already deduplicated by source's window function logic, so this might not be needed here."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Placeholder for inserting summary into snp_check_tab\n",
        "# (Not explicitly present in this ODI script)"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Mark Records for Update"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# SCEN_TASK_NO {130}\n",
        "# Mark existing records in the target as 'U' in the flow table\n",
        "target_wc_badge_details_d_df = spark.table(\"workspace.prxbi_dw.wc_badge_details_d\")\n",
        "\n",
        "DeltaTable.forName(spark, \"workspace.prxbi_dw.i_wc_badge_details_d_flow\").alias(\"t\").merge(\n",
        "    target_wc_badge_details_d_df.select(\"INTEGRATION_ID\", \"DATASOURCE_NUM_ID\").alias(\"s\"),\n",
        "    \"t.INTEGRATION_ID = s.INTEGRATION_ID AND t.DATASOURCE_NUM_ID = s.DATASOURCE_NUM_ID\"\n",
        ").whenMatchedUpdate(set={\n",
        "    \"t.IND_UPDATE\": F.lit(\"U\")\n",
        "}).execute()"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Merge into Target"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# SCEN_TASK_NO {150}, {160}\n",
        "flow_to_merge_df = spark.table(\"workspace.prxbi_dw.i_wc_badge_details_d_flow\")\n",
        "target_table = DeltaTable.forName(spark, \"workspace.prxbi_dw.wc_badge_details_d\")\n",
        "\n",
        "target_table.alias(\"t\").merge(\n",
        "    flow_to_merge_df.alias(\"s\"),\n",
        "    \"t.INTEGRATION_ID = s.INTEGRATION_ID AND t.DATASOURCE_NUM_ID = s.DATASOURCE_NUM_ID\"\n",
        ").whenMatchedUpdate(\n",
        "    condition=\"s.IND_UPDATE = 'U'\",\n",
        "    set={\n",
        "        \"t.BADGE_ID\": F.col(\"s.BADGE_ID\"),\n",
        "        \"t.BADGE_LOCATION\": F.col(\"s.BADGE_LOCATION\"),\n",
        "        \"t.BADGE_TOKEN\": F.col(\"s.BADGE_TOKEN\"),\n",
        "        \"t.BADGE_VERSION\": F.col(\"s.BADGE_VERSION\"),\n",
        "        \"t.CONTACT_EMAIL\": F.col(\"s.CONTACT_EMAIL\"),\n",
        "        \"t.CONTACT_FIRST_NAME\": F.col(\"s.CONTACT_FIRST_NAME\"),\n",
        "        \"t.CONTACT_LAST_NAME\": F.col(\"s.CONTACT_LAST_NAME\"),\n",
        "        \"t.CONTACT_JOB_TITLE\": F.col(\"s.CONTACT_JOB_TITLE\"),\n",
        "        \"t.CONTACT_PERSON_ID\": F.col(\"s.CONTACT_PERSON_ID\"),\n",
        "        \"t.CREATION_REG_TYPE\": F.col(\"s.CREATION_REG_TYPE\"),\n",
        "        \"t.CREATION_TYPE\": F.col(\"s.CREATION_TYPE\"),\n",
        "        \"t.CULTURE\": F.col(\"s.CULTURE\"),\n",
        "        \"t.CUSTOMER_TYPE\": F.col(\"s.CUSTOMER_TYPE\"),\n",
        "        \"t.EVENT_EDITION_CODE\": F.col(\"s.EVENT_EDITION_CODE\"),\n",
        "        \"t.BADGE_UPDATE_FLG\": F.col(\"s.BADGE_UPDATE_FLG\"),\n",
        "        \"t.MARKETING_PREF_PROMPT\": F.col(\"s.MARKETING_PREF_PROMPT\"),\n",
        "        \"t.ORG_NAME\": F.col(\"s.ORG_NAME\"),\n",
        "        \"t.ORG_CITY\": F.col(\"s.ORG_CITY\"),\
        "        \"t.ORG_COUNTRY\": F.col(\"s.ORG_COUNTRY\"),\n",
        "        \"t.ORG_ID\": F.col(\"s.ORG_ID\"),\n",
        "        \"t.ORG_STATE\": F.col(\"s.ORG_STATE\"),\n",
        "        \"t.PARTICIPATING_ORG_ID\": F.col(\"s.PARTICIPATING_ORG_ID\"),\n",
        "        \"t.PRODUCT_CODE\": F.col(\"s.PRODUCT_CODE\"),\n",
        "        \"t.QR_CODE\": F.col(\"s.QR_CODE\"),\n",
        "        \"t.REGISTRATION_ID\": F.col(\"s.REGISTRATION_ID\"),\n",
        "        \"t.STATUS\": F.col(\"s.STATUS\"),\n",
        "        \"t.STAFF_COMPANY_NAME\": F.col(\"s.STAFF_COMPANY_NAME\"),\n",
        "        \"t.STAFF_COMPANY_ADDR\": F.col(\"s.STAFF_COMPANY_ADDR\"),\n",
        "        \"t.STAFF_PHONE_NUM\": F.col(\"s.STAFF_PHONE_NUM\"),\n",
        "        \"t.STAFF_REPORTING\": F.col(\"s.STAFF_REPORTING\"),\n",
        "        \"t.STAFF_STANDS\": F.col(\"s.STAFF_STANDS\"),\n",
        "        \"t.STAFF_USER_ACCESS\": F.col(\"s.STAFF_USER_ACCESS\"),\n",
        "        \"t.VERSION_NUM\": F.col(\"s.VERSION_NUM\"),\n",
        "        \"t.MOBILEPHONE\": F.col(\"s.MOBILEPHONE\"),\n",
        "        \"t.FIRSTSCANNEDDATE\": F.col(\"s.FIRSTSCANNEDDATE\"),\n",
        "        \"t.LASTPRINTEDDATE\": F.col(\"s.LASTPRINTEDDATE\"),\n",
        "        \"t.FIRSTSCANNEDDATE_FLG\": F.col(\"s.FIRSTSCANNEDDATE_FLG\"),\n",
        "        \"t.LASTPRINTEDDATE_FLG\": F.col(\"s.LASTPRINTEDDATE_FLG\"),\n",
        "        \"t.ACCESSVALIDITYMODIFIEDDATE\": F.col(\"s.ACCESSVALIDITYMODIFIEDDATE\"),\n",
        "        \"t.CREATEDDATE\": F.col(\"s.CREATEDDATE\"),\n",
        "        \"t.COMPANYPRODUCTCODE\": F.col(\"s.COMPANYPRODUCTCODE\"),\n",
        "        \"t.PAYMENTSTATUS\": F.col(\"s.PAYMENTSTATUS\"),\n",
        "        \"t.PHOTOKEY\": F.col(\"s.PHOTOKEY\"),\n",
        "        \"t.PHOTOSOURCE\": F.col(\"s.PHOTOSOURCE\"),\n",
        "        \"t.PHOTOSOURCETYPE\": F.col(\"s.PHOTOSOURCETYPE\"),\n",
        "        \"t.PACKAGE_NAME\": F.col(\"s.PACKAGE_NAME\"),\n",
        "        \"t.W_UPDATE_DT\": F.current_timestamp()\n",
        "    }\n",
        ").whenNotMatchedInsert(\n",
        "    condition=\"s.IND_UPDATE = 'I'\",\n",
        "    values={\n",
        "        # ROW_WID is assumed to be GENERATED ALWAYS AS IDENTITY in Databricks and is not explicitly inserted.\n",
        "        \"BADGE_ID\": F.col(\"s.BADGE_ID\"),\n",
        "        \"BADGE_LOCATION\": F.col(\"s.BADGE_LOCATION\"),\n",
        "        \"BADGE_TOKEN\": F.col(\"s.BADGE_TOKEN\"),\n",
        "        \"BADGE_VERSION\": F.col(\"s.BADGE_VERSION\"),\n",
        "        \"CONTACT_EMAIL\": F.col(\"s.CONTACT_EMAIL\"),\n",
        "        \"CONTACT_FIRST_NAME\": F.col(\"s.CONTACT_FIRST_NAME\"),\n",
        "        \"CONTACT_LAST_NAME\": F.col(\"s.CONTACT_LAST_NAME\"),\n",
        "        \"CONTACT_JOB_TITLE\": F.col(\"s.CONTACT_JOB_TITLE\"),\n",
        "        \"CONTACT_PERSON_ID\": F.col(\"s.CONTACT_PERSON_ID\"),\n",
        "        \"CREATION_REG_TYPE\": F.col(\"s.CREATION_REG_TYPE\"),\n",
        "        \"CREATION_TYPE\": F.col(\"s.CREATION_TYPE\"),\n",
        "        \"CULTURE\": F.col(\"s.CULTURE\"),\n",
        "        \"CUSTOMER_TYPE\": F.col(\"s.CUSTOMER_TYPE\"),\n",
        "        \"EVENT_EDITION_CODE\": F.col(\"s.EVENT_EDITION_CODE\"),\n",
        "        \"BADGE_UPDATE_FLG\": F.col(\"s.BADGE_UPDATE_FLG\"),\n",
        "        \"MARKETING_PREF_PROMPT\": F.col(\"s.MARKETING_PREF_PROMPT\"),\n",
        "        \"ORG_NAME\": F.col(\"s.ORG_NAME\"),\n",
        "        \"ORG_CITY\": F.col(\"s.ORG_CITY\"),\n",
        "        \"ORG_COUNTRY\": F.col(\"s.ORG_COUNTRY\"),\n",
        "        \"ORG_ID\": F.col(\"s.ORG_ID\"),\n",
        "        \"ORG_STATE\": F.col(\"s.ORG_STATE\"),\n",
        "        \"PARTICIPATING_ORG_ID\": F.col(\"s.PARTICIPATING_ORG_ID\"),\n",
        "        \"PRODUCT_CODE\": F.col(\"s.PRODUCT_CODE\"),\n",
        "        \"QR_CODE\": F.col(\"s.QR_CODE\"),\n",
        "        \"REGISTRATION_ID\": F.col(\"s.REGISTRATION_ID\"),\n",
        "        \"STATUS\": F.col(\"s.STATUS\"),\n",
        "        \"STAFF_COMPANY_NAME\": F.col(\"s.STAFF_COMPANY_NAME\"),\n",
        "        \"STAFF_COMPANY_ADDR\": F.col(\"s.STAFF_COMPANY_ADDR\"),\n",
        "        \"STAFF_PHONE_NUM\": F.col(\"s.STAFF_PHONE_NUM\"),\n",
        "        \"STAFF_REPORTING\": F.col(\"s.STAFF_REPORTING\"),\n",
        "        \"STAFF_STANDS\": F.col(\"s.STAFF_STANDS\"),\n",
        "        \"STAFF_USER_ACCESS\": F.col(\"s.STAFF_USER_ACCESS\"),\n",
        "        \"VERSION_NUM\": F.col(\"s.VERSION_NUM\"),\n",
        "        \"INTEGRATION_ID\": F.col(\"s.INTEGRATION_ID\"),\n",
        "        \"DATASOURCE_NUM_ID\": F.col(\"s.DATASOURCE_NUM_ID\"),\n",
        "        \"MOBILEPHONE\": F.col(\"s.MOBILEPHONE\"),\n",
        "        \"FIRSTSCANNEDDATE\": F.col(\"s.FIRSTSCANNEDDATE\"),\n",
        "        \"LASTPRINTEDDATE\": F.col(\"s.LASTPRINTEDDATE\"),\n",
        "        \"FIRSTSCANNEDDATE_FLG\": F.col(\"s.FIRSTSCANNEDDATE_FLG\"),\n",
        "        \"LASTPRINTEDDATE_FLG\": F.col(\"s.LASTPRINTEDDATE_FLG\"),\n",
        "        \"ACCESSVALIDITYMODIFIEDDATE\": F.col(\"s.ACCESSVALIDITYMODIFIEDDATE\"),\n",
        "        \"CREATEDDATE\": F.col(\"s.CREATEDDATE\"),\n",
        "        \"COMPANYPRODUCTCODE\": F.col(\"s.COMPANYPRODUCTCODE\"),\n",
        "        \"PAYMENTSTATUS\": F.col(\"s.PAYMENTSTATUS\"),\n",
        "        \"PHOTOKEY\": F.col(\"s.PHOTOKEY\"),\n",
        "        \"PHOTOSOURCE\": F.col(\"s.PHOTOSOURCE\"),\n",
        "        \"PHOTOSOURCETYPE\": F.col(\"s.PHOTOSOURCETYPE\"),\n",
        "        \"PACKAGE_NAME\": F.col(\"s.PACKAGE_NAME\"),\n",
        "        \"W_INSERT_DT\": F.current_timestamp(),\n",
        "        \"W_UPDATE_DT\": F.current_timestamp()\n",
        "    }\n",
        ").execute()"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Optimize Target"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# SCEN_TASK_NO {170} (Implicit COMMIT), {190} (Implicit DBMS_STATS on target if exists)\n",
        "spark.sql(\"SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false\")\n",
        "spark.sql(\"OPTIMIZE workspace.prxbi_dw.wc_badge_details_d ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID)\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Cleanup"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# SCEN_TASK_NO {180}, {210}\n",
        "spark.sql(\"DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow PURGE\")\n",
        "spark.sql(\"DROP TABLE IF EXISTS workspace.prxbi_dw.c_mercury_badge_ts_stg PURGE\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Validation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "final_target_count = spark.table(\"workspace.prxbi_dw.wc_badge_details_d\").count()\n",
        "print(f\"Final target table count: {final_target_count}\")\n",
        "\n",
        "print(\"Sample of final target data:\")\n",
        "display(spark.table(\"workspace.prxbi_dw.wc_badge_details_d\").limit(10))\n",
        "\n",
        "# Placeholder for error summary if E$ table was populated\n",
        "# error_count = spark.table(\"workspace.prxbi_dw.e_some_error_table\").filter(F.col(\"ODI_SESS_NO\") == F.lit(odi_sess_no)).count()\n",
        "# print(f\"Number of errors recorded for this session: {error_count}\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "spark.stop()"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Conversion Notes\n",
        "\n",
        "1.  **ROW_WID Handling:** The `ROW_WID` column in the target table `workspace.prxbi_dw.wc_badge_details_d` is derived from an Oracle `SEQUENCE.NEXTVAL`. In Databricks, it is assumed this column is configured as `GENERATED ALWAYS AS IDENTITY`. Consequently, `ROW_WID` is excluded from the `set` clause in `whenMatchedUpdate` and `values` clause in `whenNotMatchedInsert` as it is automatically managed by the Delta table.\n",
        "2.  **Schema and Table Naming:** Oracle schemas `PRXBI_DW_SEP` and `PRXBI_TS_SEP` have been mapped to `workspace.prxbi_dw` and `workspace.prxbi_ts` respectively. Staging and flow tables have been given descriptive names (e.g., `C$_0A7SUCRIPSM1CG2656H955OU5QP` -> `c_mercury_badge_ts_stg`).\n",
        "3.  **Deduplication Logic:** The ODI `INNER JOIN` with `MAX()` aggregates (SCEN_TASK_NO {50}) and `rank() over()` (SCEN_TASK_NO {100} subquery) have been converted to PySpark `Window.partitionBy().orderBy().row_number().filter(F.col(\"rn\") == 1)` as per the migration guidelines.\n",
        "4.  **ETL Parameters:** Global parameters (`#GLOBAL.v_ETL_JOB_TYPE`, etc.) are accessed via `dbutils.widgets.get()` and assumed to be provided at notebook execution. `etl_last_extract_time`, `etl_current_extract_time`, and `etl_row_wid` are dynamically retrieved from `workspace.prxbi_dw.wc_etl_parameters`.\n",
        "5.  **Error/Audit Tables:** Placeholders for `E$_*` and `SNP_CHECK_TAB` have been included, but no specific logic for populating them was present in the provided ODI script. If error handling or auditing is required, these sections would need to be implemented.\n",
        "6.  **`DATASOURCE_NUM_ID` Type:** The `DATASOURCE_NUM_ID` column in the flow table is `VARCHAR2(10 CHAR)` in the ODI DDL but `380` suggests an integer. It has been cast to `StringType()` during flow table creation to align with the DDL, while the widget value is `int` for flexibility.\n",
        "7.  **`NULL` Comparison (`NOT EXISTS`):** The explicit `((T.COL = S.COL) OR (T.COL IS NULL AND S.COL IS NULL))` conditions for `NOT EXISTS` in ODI are implicitly handled by Spark's default join behavior for equality conditions, which treats `NULL = NULL` as `TRUE` when used in join predicates for `DeltaTable.merge()`."
      ]
    }
  ]
}